<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Test the IVR like a product

**[Test the IVR like a product](https://learning.nextia-ai.com/courses/speech/m07/test-the-ivr-like-a-product/)** · Speech AI: Transcribe, Understand and Speak · Module 7, lesson 6 of 6 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn the IVR's call records into the measures a product team reads (passed, containment, time to the answer), read them by kind and by line without being fooled by the mix, make a test stricter, and find a bug in the test harness itself.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | The four measures of an IVR, from [What an IVR is, and why companies still use one](https://learning.nextia-ai.com/courses/speech/m07/what-an-ivr-is/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m07/test-the-ivr-like-a-product/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M07-L06-test-the-ivr-like-a-product/test-the-ivr-like-a-product-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 1 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/recordings/agent/ivr_replay.jsonl": "d4d8dd6de71037e7ab6c2ca0433214ec3cfec5112a2de42283c4933994372438",
    "data/ivr/calls.jsonl": "5ee1fbad92c79b73fa908076325f2ce6d4030ed12439dacc26b6dcd9bc00d828",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The call records

The course replayed the 60 public IVR test calls through the IVR (Whisper base, end of turn 0.8 s) and kept every call record: the outcome, what the IVR heard, the prompts it played, a timeline, and the expected result. Run the next cell.

In [ ]:
records = [json.loads(line) for line in open("data/recordings/agent/ivr_replay.jsonl", encoding="utf-8")
           if line.strip()]
ivr_calls = {}
for line in open("data/ivr/calls.jsonl", encoding="utf-8"):
    row = json.loads(line)
    ivr_calls[row["call_id"]] = row
r = records[0]
print(len(records), "call records. The first one:")
print({k: r[k] for k in ("call_id", "kind", "outcome", "handoff_reason", "heard", "prompts", "seconds", "passed")})
print("expected:", r["expected"])

## 2. The measures

> **Predict.** How many of the 60 calls should the IVR answer by itself? That number is the best possible containment.

In [ ]:
CONTAINED = ("status_given", "hours_given", "return_started")

def measures(rows):
    n = len(rows)
    contained = [r for r in rows if r["outcome"] in CONTAINED]
    return {"calls": n, "passed": sum(r["passed"] for r in rows),
            "containment": round(len(contained) / n, 3),
            "to a person": round(sum(r["outcome"] == "handover" for r in rows) / n, 3),
            "wrong intent": round(sum(r["wrong_intent"] for r in rows) / n, 3),
            "s to answer": round(sum(r["seconds"] for r in contained) / len(contained), 1) if contained else None}

overall = measures(records)
print(overall)
should_answer = sum(r["expected"]["outcome"] in CONTAINED for r in records)
print("calls that should be answered by the IVR:", should_answer, "-> best possible containment",
      round(should_answer / len(records), 3))

> **Check.** 39 passed, containment 35%, time to the answer 50.2 seconds. 41 calls should be answered by the IVR: no IVR can contain more than that on this set.

## 3. By kind

In [ ]:
def by(key):
    groups = {}
    for r in records:
        groups.setdefault(r[key], []).append(r)
    print(f'{key:16} {"calls":>5} {"passed":>6} {"contained":>9} {"s to answer":>11}')
    for k, rows in sorted(groups.items()):
        m = measures(rows)
        s = "-" if m["s to answer"] is None else m["s to answer"]
        print(f'{k:16} {m["calls"]:>5} {m["passed"]:>6} {m["containment"]:>9.0%} {s:>11}')

by("kind")

The failures gather in the kinds where the caller **says an order number**. The keypad kinds, the opening hours and every way to a person pass.

## 4. By line, and the mix inside each slice

> **Predict.** The landline is the cleanest line. Will it have the highest containment?

In [ ]:
by("line")
for line in ("landline", "mobile", "voip_g726"):
    rows = [r for r in records if r["line"] == line]
    must_person = sum(r["expected"]["outcome"] == "handover" for r in rows)
    should = [r for r in rows if r["expected"]["outcome"] in CONTAINED]
    got = sum(r["outcome"] in CONTAINED for r in should)
    print(f"{line:10} {len(rows)} calls, {must_person} should reach a person; "
          f"contained {got} of the {len(should)} that should be")

> **Check.** The landline has the lowest containment, but it also holds most of the calls that should reach a person. Among the calls that **should** be contained, the lines are much closer. Always look at what is inside a slice before you blame the thing that names it.

## 5. A stricter test

The replay's rule checks the outcome (and the order number). For a handover, it does not check the **reason**.

> **Predict.** If the reason must match too, how many calls still pass?

In [ ]:
def passed_strict(r):
    """Like the replay's rule, and for a handover the reason must be the expected one too."""
    if not r["passed"]:
        return False
    if r["outcome"] == "handover" and r["expected"].get("handoff_reason"):
        return r["handoff_reason"] == r["expected"]["handoff_reason"]
    return True

strict = [r["call_id"] for r in records if passed_strict(r)]
loose_only = [(r["call_id"], r["kind"], r["handoff_reason"], r["expected"]["handoff_reason"])
              for r in records if r["passed"] and r["call_id"] not in strict]
print(f"passed: {sum(r['passed'] for r in records)}; with the reason checked too: {len(strict)}")
for row in loose_only:
    print("  passed for another reason:", row)

> **Check.** 33 instead of 39: 6 calls reached a person for another reason than the test expected. Most are return calls that ended with `no_input`, before the IVR could even look at the order. They "passed" because a person took over in the end.

## 6. Your turn: a bug in the test harness

In a keypad call, every key is right, but the replay plays a retry ("Sorry, I did not get that.") right after the next prompt. Look at I-017 first:

In [ ]:
for t, what, detail in next(r for r in records if r["call_id"] == "I-017")["timeline"]:
    if what in ("say", "key"):
        print(f"{t:6.1f} s  {what:4} {detail[:70]}")

> **Your turn.** Write `retry_after_key(record)`: `True` when the timeline has a key event, then a prompt, then at once a **retry** prompt, with nothing heard in between. The retry prompts start with "Sorry, I did not get that.", "That number does not look right.", "Please say yes or no", and the French "Désolé, je n'ai pas compris.", "Ce numéro ne semble pas correct.", "Dites oui ou non". Use only the events `say`, `key` and `heard`.

In [ ]:
RETRY_STARTS = ("Sorry, I did not get that.", "That number does not look right.", "Please say yes or no",
                "Désolé, je n'ai pas compris.", "Ce numéro ne semble pas correct.", "Dites oui ou non")

def retry_after_key(record):
    """A key, the next prompt at once, then a retry prompt with nothing heard in between."""
    events = [e for e in record["timeline"] if e[1] in ("say", "key", "heard")]
    for a, b, c in zip(events, events[1:], events[2:]):
        if a[1] == "key" and b[1] == "say" and c[1] == "say" and c[2].startswith(RETRY_STARTS):
            return True
    return False

retry_calls = sorted(r["call_id"] for r in records if retry_after_key(r))
print(len(retry_calls), retry_calls[:8])

In [ ]:
expect_hash('the calls with a retry right after a key', retry_calls, '197586b4c6a3d924')

> **Check.** 30 calls. The cause is in the replay, not in the IVR: after a key press, the replay kept the rest of the key's recording (the tone's tail and silence) as the caller's next turn, and counted a no-input. The course tested a fix: no pass and no containment changed, but a keypad caller gets the status about 30 seconds sooner (67.8 s against 38.1 s for status by keypad). A test harness is code too.

## Next

You read the IVR's results like a product team, made the test stricter, and found a bug in the harness. The [lesson page](https://learning.nextia-ai.com/courses/speech/m07/test-the-ivr-like-a-product/) has the hidden calls, the test phone and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Where a menu stops working](https://learning.nextia-ai.com/courses/speech/m08/where-a-menu-stops-working/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m07/test-the-ivr-like-a-product/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.